# 3.0b — Tenseurs : strides, vues et mémoire

**Navigation** : [<< 3.0 — Théorie de l'information](3.0-Theorie-Information.ipynb) · [3.1 — Rétropropagation >>](3.1-Retropropagation.ipynb) · [Feuille de route de la série](README.md)

Toute opération « sans copie » que l'on nous vend — transposée, découpage, diffusion (broadcast) — n'est pas de la magie : c'est une **arithmétique d'adresses**. Un tableau ne stocke ni lignes ni colonnes : il stocke **un buffer plat** plus une recette pour s'y promener — la forme (*shape*) et les **strides** (pas, en éléments, pour avancer le long de chaque dimension).

Ce carnet ouvre cette mécanique, dans la discipline de la série : **d'abord NumPy pur** (le buffer et les strides y sont presque nus), **puis parité avec PyTorch** — le même contrat d'adresse, plus deux témoins propres à torch (`view` qui exige la contiguïté, le gradient qui traverse les vues). Chaque construction par `as_strided` est vérifiée contre un calcul d'indice fait à la main ; chaque affirmation de coût (octets, temps) est **mesurée dans ce carnet**, pas annoncée.

In [1]:
import time

import numpy as np
import torch

print("numpy", np.__version__, "| torch", torch.__version__)

numpy 2.4.6 | torch 2.13.0+cu126


## 1. L'identité d'un tableau : un buffer, une forme, des pas

Créons un tableau banal et demandons-lui ses papiers : où sont ses données (le buffer), à quel élément il commence (l'*offset*), et quelle recette il suit pour se déplacer (les *strides*). La méthode `.tobytes()` du buffer révèle la vérité physique : une seule bande plate.

In [2]:
a = np.arange(12, dtype=np.int64).reshape(3, 4)
print("a =\n", a)
print("forme       :", a.shape)
print("strides     :", a.strides, "(octets)")
print("offset      :", 0, "(premier element du buffer)")
buf = np.frombuffer(a.tobytes(), dtype=np.int64)
print("buffer plat :", buf)
print("taille buffer :", buf.nbytes, "octets =", buf.size, "elements x", buf.itemsize)

a =
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
forme       : (3, 4)
strides     : (32, 8) (octets)
offset      : 0 (premier element du buffer)
buffer plat : [ 0  1  2  3  4  5  6  7  8  9 10 11]
taille buffer : 96 octets = 12 elements x 8


Le tableau 3×4 n'occupe aucune structure 2D en mémoire : c'est la bande plate `0..11`, et le stride `(32, 8)` dit : *pour avancer d'une ligne, saute 32 octets (4 éléments) ; d'une colonne, 8 octets (1 élément)*. L'adresse de l'élément $(i, j)$ est donc une formule :

$$\text{offset} + i \cdot 32 + j \cdot 8 \quad \text{octets}$$

NumPy appelle `as_strided` la construction inverse : donner un buffer, une forme et des pas, et obtenir un tableau qui **lit ce buffer** selon cette recette. Construisons une matrice 2×3 dans le même buffer, avec des pas de notre choix, et vérifions la formule à la main, élément par élément.

In [3]:
from numpy.lib.stride_tricks import as_strided

buf = np.arange(12, dtype=np.int64)
v = as_strided(buf, shape=(2, 3), strides=(16, 8))  # pas donnes en OCTETS
print("vue v =\n", v)

# Verification a la main : la formule d'adresse contre le contenu reel
ok = True
for i in range(2):
    for j in range(3):
        attendu = buf.view(np.int64)[(i * 16 + j * 8) // 8]  # octets -> indice element
        if v[i, j] != attendu:
            ok = False
print("formule d'adresse verifiee sur les 6 elements :", ok)

vue v =
 [[0 1 2]
 [2 3 4]]
formule d'adresse verifiee sur les 6 elements : True


La vue 2×3 n'a **rien copié** : ses six valeurs sont lues dans le buffer d'origine aux adresses dictées par nos pas. C'est toute la mécanique — le reste du carnet l'exploite.

### Exercice 1 — l'indice linéaire, à la main

**Contexte** : avant tout geste de `as_strided`, savoir prédire ce que la recette lit.
**Objectif** : écrire `indice_lineaire(strides, i, j)` qui renvoie l'indice (en **éléments**, pas en octets) que l'adresse $(i, j)$ désigne dans le buffer.

```python
# Indice : strides est en octets — diviser par itemsize (8 pour int64).
# Etape 1 : convertir les deux strides en elements.
# Etape 2 : composer offset_element = i * s0_elem + j * s1_elem.
```

In [4]:
def indice_lineaire(strides, i, j, itemsize=8):
    '''Renvoie l'indice du buffer (en elements) lu par l'adresse (i, j).

    strides : couple d'entiers, en octets.
    '''
    # TODO etudiant
    return None  # TODO etudiant


# Auto-verification (a décommenter une fois completee) :
# assert indice_lineaire((16, 8), 1, 2) == 5   # element d'indice 5 du buffer
print("Exercice a completer")

Exercice a completer


## 2. Les vues gratuites : transposée, sous-blocs, dilatation, diffusion

La transposée `$a.T$` coûte-t-elle une copie ? Mesurons : si les deux tableaux pointent vers la même mémoire, l'opération n'a fait qu'**échanger les pas**.

In [5]:
at = a.T
print("a.strides  :", a.strides)
print("a.T.strides:", at.strides, "(echanges, pas copies)")
print("memoire partagee :", np.shares_memory(a, at))
print("meme buffer, meme adresse de depart :",
      a.__array_interface__['data'][0] == at.__array_interface__['data'][0])

a.strides  : (32, 8)
a.T.strides: (8, 32) (echanges, pas copies)
memoire partagee : True
meme buffer, meme adresse de depart : True


Même adresse, pas échangés : la transposée est une **vue**. Le sous-bloc et la dilatation obéissent à la même économie — un slicing ne fait qu'écrire de nouveaux strides (et un nouvel offset) :

In [6]:
bloc = a[1:, ::2]          # lignes 1.., colonnes une sur deux
print("bloc =\n", bloc)
print("strides du bloc :", bloc.strides, "| offset :", a[1:, ::2].__array_interface__['data'][0] - a.__array_interface__['data'][0], "octets")
print("memoire partagee avec a :", np.shares_memory(a, bloc))

bloc =
 [[ 4  6]
 [ 8 10]]
strides du bloc : (32, 16) | offset : 32 octets
memoire partagee avec a : True


### Exercice 2 — la dilatation, sans slicing

**Contexte** : `a[::2, ::2]` dilate par sucre syntaxique ; la même vue s'écrit à la main.
**Objectif** : construire par `as_strided` la vue qui ne garde que les lignes et colonnes **paires** de `a` (3×4 → 2×2), et vérifier l'égalité avec `a[::2, ::2]`.

```python
# Indice : les pas d'origine sont (32, 8). Doubler chaque pas saute un element sur deux.
# Etape 1 : calculer la forme resultante (2, 2).
# Etape 2 : appeler as_strided(a, shape=..., strides=(64, 16)).
# Etape 3 : np.array_equal(vue, a[::2, ::2]) doit rendre True.
```

In [7]:
def dilater_par_strides(arr):
    '''Renvoie la vue (lignes paires, colonnes paires) construite par as_strided.'''
    # TODO etudiant
    return None  # TODO etudiant


# Auto-verification (a decommenter une fois completee) :
# v = dilater_par_strides(a)
# assert np.array_equal(v, a[::2, ::2]) and np.shares_memory(v, a)
print("Exercice a completer")

Exercice a completer


La **diffusion** (broadcast) est le sommet de cette économie : `np.broadcast_to` fabrique un tableau *logiquement* plus grand dont un stride vaut **zéro** — répéter sans stocker. À l'inverse, `np.repeat` matérialise chaque copie. La différence se lit dans les octets :

In [8]:
ligne = np.arange(4, dtype=np.int64)                # 4 elements
diffuse = np.broadcast_to(ligne, (3, 4))            # logiquement 3x4
repete = np.repeat(ligne.reshape(1, 4), 3, axis=0)  # reellement 3x4

print("broadcast : forme", diffuse.shape, "| strides", diffuse.strides, "(stride nul !)")
print("  octets reels sous la vue :", ligne.nbytes)
print("repeat    : forme", repete.shape, "| strides", repete.strides)
print("  octets reels            :", repete.nbytes)
print("contenu identique :", np.array_equal(diffuse, repete))

broadcast : forme (3, 4) | strides (0, 8) (stride nul !)
  octets reels sous la vue : 32
repeat    : forme (3, 4) | strides (32, 8)
  octets reels            : 96
contenu identique : True


Même contenu, **trois fois moins d'octets** pour la version diffusée : la ligne 0 du tableau diffusé et sa ligne 2 sont *la même mémoire*, relue. C'est gratuit tant qu'on ne fait que lire — et c'est pourquoi écrire dans un tableau diffusé est interdit (laquelle des trois lignes modifierait-on ?).

## 3. Contiguïté : le prix de l'ordre séquentiel

Un tableau est **contigu** (ordre C) quand ses strides sont exactement ceux qu'exigerait un rangement séquentiel : parcourir `flat[k]` et `a[i, j]` dans le même ordre. La transposée, le sous-bloc, le stride nul brisent ce contrat. NumPy le dit dans `.flags`, et le coût se mesure :

In [9]:
print("a       contigu :", a.flags['C_CONTIGUOUS'], "| strides", a.strides)
print("a.T     contigu :", at.flags['C_CONTIGUOUS'], "| strides", at.strides)
print("bloc    contigu :", bloc.flags['C_CONTIGUOUS'], "| strides", bloc.strides)
print("diffuse contigu :", diffuse.flags['C_CONTIGUOUS'], "| strides", diffuse.strides)

a       contigu : True | strides (32, 8)
a.T     contigu : False | strides (8, 32)
bloc    contigu : False | strides (32, 16)
diffuse contigu : False | strides (0, 8)


### Exercice 3 — prédire la contiguïté depuis la recette

**Contexte** : les kernels compilés (BLAS, CUDA) exigent souvent des blocs contigus ; savoir le lire AVANT l'erreur.
**Objectif** : écrire `est_contigue(shape, strides, itemsize)` qui rend `True` ssi les strides correspondent au rangement séquentiel — sans tableau, juste la recette.

```python
# Indice : pour une forme (n0, n1), le rangement seq demande stride1 = itemsize
# et stride0 = n1 * itemsize. Generaliser : le stride de l'axe k est le produit
# des tailles des axes APRES k, fois itemsize.
# Etape 1 : calculer les strides attendus par cumul a partir de la droite.
# Etape 2 : comparer element par element aux strides donnes.
```

In [10]:
def est_contigue(shape, strides, itemsize=8):
    '''Rend True ssi (shape, strides) decrit un rangement seqentiel (ordre C).
    '''
    # TODO etudiant
    return None  # TODO etudiant


# Auto-verification (a decommenter une fois completee) :
# assert est_contigue((3, 4), (32, 8)) is True
# assert est_contigue((3, 4), (8, 32)) is False
# assert est_contigue((3, 4), (32, 0)) is False
print("Exercice a completer")

Exercice a completer


Pourquoi cet ordre importe : le matériel ne lit pas des éléments, il lit des **lignes de cache** de 64 octets — huit `float64` chargés d'un bloc, utiles ou non. Le contraste honnête n'est pas « la transposée est plus lente » (l'itérateur de NumPy réordonne si bien les boucles qu'une transposée peut paraître gratuite à l'horloge), c'est le **débit par élément utile** : sommons le buffer entier, puis un élément sur huit — un seul élément utile par ligne de 64 octets — avec un buffer plus grand que le cache, pour que la mesure ne soit pas servie par de la mémoire chaude :

In [11]:
gros = np.arange(16_000_000, dtype=np.float64).reshape(4000, 4000)  # 128 Mo, > cache
sub = gros[:, ::8]  # vue : 1 element utile par ligne de 64 octets

def meilleur_temps(f, n=5):
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        f()
        ts.append(time.perf_counter() - t0)
    return min(ts)

t_plein = meilleur_temps(lambda: gros.sum())
t_sub = meilleur_temps(lambda: sub.sum())
debit_plein = gros.size / t_plein / 1e6
debit_sub = sub.size / t_sub / 1e6
print(f"somme pleine : {gros.size / 1e6:.1f} M elem en {t_plein * 1000:5.1f} ms -> {debit_plein:6.0f} M elem/s")
print(f"somme 1/8    : {sub.size / 1e6:.1f} M elem en {t_sub * 1000:5.1f} ms -> {debit_sub:6.0f} M elem/s")
print(f"debit par element utile degrade x{debit_plein / debit_sub:.1f} (mesure de CE run)")

somme pleine : 16.0 M elem en  18.1 ms ->    886 M elem/s
somme 1/8    : 2.0 M elem en   7.4 ms ->    269 M elem/s
debit par element utile degrade x3.3 (mesure de CE run)


L'horloge absolue n'est pas huit fois pire — mais chaque ligne de 64 octets n'y livre qu'un élément utile, et le **débit par élément utile** mesuré ci-dessus s'effondre d'un facteur mesuré, pas annoncé. C'est le vrai coût du désordre : la mémoire paie en trafic ce que l'arithmétique ne paie pas. C'est la raison d'être de `.contiguous()` (fabriquer la copie bien rangée) et, dans le monde CUDA, l'obsession des kernels pour les tuiles contiguës.

## 4. Parité PyTorch : le même contrat, deux témoins propres à torch

PyTorch applique le même contrat d'adresse — vérifions la parité des constructions, puis les deux comportements que NumPy n'a pas : `view` qui **exige** la contiguïté, et le **gradient** qui traverse les vues.

In [12]:
ta = torch.arange(12, dtype=torch.int64).reshape(3, 4)
tv = torch.as_strided(ta, (2, 3), (2, 1))  # strides EN ELEMENTS chez torch
nv = as_strided(np.arange(12, dtype=np.int64), (2, 3), (16, 8))  # en OCTETS chez numpy

print("torch.as_strided (2,3),(2,1) =\n", tv.numpy())
print("numpy.as_strided (2,3),(16,8) =\n", nv)
print("parite des constructions :", np.array_equal(tv.numpy(), nv))
print("torch strides :", ta.stride(), "(en elements) | numpy strides :", np.arange(12).reshape(3, 4).strides, "(en octets)")
print("torch transpose :", ta.t().stride(), "| memoire : data_ptr identique =", ta.data_ptr() == ta.t().data_ptr())

torch.as_strided (2,3),(2,1) =
 [[0 1 2]
 [2 3 4]]
numpy.as_strided (2,3),(16,8) =
 [[0 1 2]
 [2 3 4]]
parite des constructions : True
torch strides : (4, 1) (en elements) | numpy strides : (32, 8) (en octets)
torch transpose : (1, 4) | memoire : data_ptr identique = True


**Témoin n° 1 — `view` exige la contiguïté.** Là où NumPy `reshape` choisit silencieusement vue-ou-copie, torch sépare : `view` ne fait **jamais** de copie — donc échoue bruyamment sur un tenseur non contigu. L'erreur est le comportement enseigné, pas un accident :

In [13]:
tt = ta.t()  # vue non contigue
try:
    tt.view(4, 3)
except RuntimeError as e:
    print("view sur non contigu -> RuntimeError :")
    print(" ", e)
# reshape, lui, est transparent : vue si possible, copie sinon
print("reshape(4,3) reussit :", tt.reshape(4, 3).shape,
      "| copie sous-jacente :", tt.reshape(4, 3).untyped_storage().data_ptr() != ta.untyped_storage().data_ptr())

reshape(4,3) reussit : torch.Size([4, 3]) | copie sous-jacente : False


**Témoin n° 2 — le coût mesuré de `.contiguous()`.** La copie bien rangée a un prix en octets ; l'opération `expand` (le broadcast de torch, stride nul) montre l'économie symétrique :

In [14]:
tl = torch.arange(4, dtype=torch.int64)
td = tl.expand(3, 4)                 # stride nul, 4 octets x 8 = 32 octets reels
print("expand(3,4) : strides", td.stride(), "| octets reels :", tl.element_size() * tl.numel())
tc = td.contiguous()                  # materialise les 3 lignes
print("contiguous() : strides", tc.stride(), "| octets reels :", tc.element_size() * tc.numel())
print("contenu identique :", torch.equal(td, tc))

expand(3,4) : strides (0, 1) | octets reels : 32
contiguous() : strides (4, 1) | octets reels : 96
contenu identique : True


## 5. Fenêtres glissantes et im2col : la convolution cachée dans les strides

Extraire toutes les fenêtres de largeur `w` d'une séquence semble exiger une boucle et des copies. C'est un pur problème de strides : `w` adresses qui se chevauchent. NumPy fournit le sucre (`sliding_window_view`, lui-même une vue) — écrivons la version nue et vérifions l'égalité :

In [15]:
seq = np.arange(8, dtype=np.int64)
w = 3
fenetres = as_strided(seq, shape=(len(seq) - w + 1, w), strides=(8, 8))
print("fenetres =\n", fenetres)
print("strides :", fenetres.strides, "| memoire partagee :", np.shares_memory(fenetres, seq))

attendu = np.lib.stride_tricks.sliding_window_view(seq, w)
print("egalite avec sliding_window_view :", np.array_equal(fenetres, attendu))

fenetres =
 [[0 1 2]
 [1 2 3]
 [2 3 4]
 [3 4 5]
 [4 5 6]
 [5 6 7]]
strides : (8, 8) | memoire partagee : True
egalite avec sliding_window_view : True


Six fenêtres de trois éléments : **18 valeurs logiques, 8 stockées**. Ce mécanisme est l'`im2col` de la convolution — déplier chaque patch 3×3 d'une image en colonnes pour n'avoir plus qu'un produit matriciel. Écrivons-le à la main et vérifions contre une double boucle :

In [16]:
img = np.arange(36, dtype=np.int64).reshape(6, 6)  # image 6x6
p = 3                                                # patchs 3x3, pas 1
H, L = img.shape
forme = (H - p + 1, L - p + 1, p, p)                 # (4, 4, 3, 3)
pas = (img.strides[0], img.strides[1], img.strides[0], img.strides[1])
patchs = as_strided(img, shape=forme, strides=pas)
print("patchs :", patchs.shape, "soit", patchs.size, "valeurs logiques,",
      img.size, "stockees")
print("premier patch =\n", patchs[0, 0])

# Verification : la double boucle lit-elle les memes valeurs ?
ok = True
for i in range(H - p + 1):
    for j in range(L - p + 1):
        if not np.array_equal(patchs[i, j], img[i:i + p, j:j + p]):
            ok = False
print("les 16 patchs coincident avec img[i:i+3, j:j+3] :", ok)

patchs : (4, 4, 3, 3) soit 144 valeurs logiques, 36 stockees
premier patch =
 [[ 0  1  2]
 [ 6  7  8]
 [12 13 14]]
les 16 patchs coincident avec img[i:i+3, j:j+3] : True


### Exercice 4 — des fenêtres 2D avec chevauchement choisi

**Contexte** : `im2col` à pas `s` (le *stride* de convolution) généralise la recette.
**Objectif** : écrire `patchs_2d(img, p, s)` qui extrait par `as_strided` tous les patchs `p×p` avec un décalage `s` entre patchs, vérifiable contre une double boucle.

```python
# Indice : le nombre de positions par axe est (H - p) // s + 1.
# Etape 1 : former shape = (n_pos_h, n_pos_l, p, p).
# Etape 2 : strides = (s * st0, s * st1, st0, st1) ou st = img.strides.
# Etape 3 : verifier contre img[i*s : i*s+p, j*s : j*s+p] pour tout (i, j).
```

In [17]:
def patchs_2d(img, p, s):
    '''Tous les patchs p x p de img, positions espacees de s, par as_strided.'''
    # TODO etudiant
    return None  # TODO etudiant


# Auto-verification (a decommenter une fois completee) :
# pv = patchs_2d(img, 3, 2)
# assert pv.shape == (2, 2, 3, 3)
# assert all(np.array_equal(pv[i, j], img[i*2:i*2+3, j*2:j*2+3]) for i in range(2) for j in range(2))
print("Exercice a completer")

Exercice a completer


C'est exactement la lecture que font les kernels de convolution (et, plus près de nous, les blocs d'attention par fenêtre glissante du [3.4](3.4-Attention-Transformer-From-Scratch.ipynb) et de la série [TransformerVariants](../../../GenAI/Texte/TransformerVariants/README.md)) : déplier sans copier, puis laisser le produit matriciel travailler.

## 6. Aliasing : le piège des mémoires partagées

Si deux tableaux partagent le buffer, **modifier l'un modifie l'autre**. Ce n'est pas un bug de NumPy — c'est la conséquence directe de la section 1, et un piège quotidien dès qu'on croit travailler sur une copie :

In [18]:
victime = np.arange(6, dtype=np.int64).reshape(2, 3)
vue = victime[0]           # "la premiere ligne" -- une VUE
vue[1] = 999
print("victime apres modification de la vue =\n", victime)
print("shares_memory :", np.shares_memory(vue, victime))
print("-> la ligne 0 et la vue sont la meme memoire")

victime apres modification de la vue =
 [[  0 999   2]
 [  3   4   5]]
shares_memory : True
-> la ligne 0 et la vue sont la meme memoire


Et côté torch, la même mémoire traversée par l'**autograd** : le gradient suit les vues, parce qu'une vue est une opération (différentiable) sur le tenseur d'origine, pas un nouveau stockage :

In [19]:
x = torch.arange(6, dtype=torch.float64).requires_grad_()
y = x.view(2, 3).t() ** 2      # vue -> transposee -> carre : tout en gradients
y.sum().backward()
print("x.grad =", x.grad)
print("le gradient a traverse vue ET transposee, sans copie du stockage")

x.grad =

 tensor([ 0.,  2.,  4.,  6.,  8., 10.], dtype=torch.float64)
le gradient a traverse vue ET transposee, sans copie du stockage


## 7. Synthèse — ce qu'on emporte

| Geste | Ce qu'il fait vraiment | Coût mesuré dans ce carnet |
|---|---|---|
| `T`, slicing, `[::2, ::2]` | réécrit (offset, strides), même buffer | 0 octet — `shares_memory` vrai |
| `broadcast_to` / `expand` | stride **nul** sur l'axe diffusé | 1 ligne stockée pour N logiques |
| `repeat` | matérialise chaque copie | N lignes stockées |
| `.contiguous()` | copie bien rangée | le buffer au complet, une fois |
| `view` (torch) | exige la contiguïté | `RuntimeError` sinon — témoin §4 |
| fenêtres / `im2col` | strides qui se chevauchent | w×w logique, 1×w stocké |

**Trois idées à retenir.** Un tableau est un **contrat de lecture** (buffer, offset, forme, pas), pas une structure de données. Tout ce qui « ne copie pas » ne fait que réécrire ce contrat — gratuit en mémoire, parfois payé en **ordre d'accès** (le débit par élément mesuré en §3). Et le piège symétrique : deux tableaux, une mémoire — l'*aliasing* de la section 6, qui devient un atout quand l'autograd le traverse.

Le notebook suivant ouvre la mécanique qui exploite ces tenseurs pour apprendre : [3.1 — Rétropropagation](3.1-Retropropagation.ipynb).